In [1]:
import os
import requests
from faker import Faker
import re
import random
from genpw import pronounceable_passwd

In [2]:
# List of common Michigan area codes
michigan_area_codes = ["248", "313", "517", "586", "616", "734", "810", "906", "947"]

# Generate a fake phone number with a Michigan area code
area_code = random.choice(michigan_area_codes)
phone_number = f"{area_code}{random.randint(100,999)}{random.randint(1000,9999)}"  # Ensures a 10-digit number

fake = Faker()

In [73]:
import requests

class Meijer:
    BASE_URL = "https://api.meijer.com"
    AUTH_URL = "https://id.meijer.com"
    
    SUBSCRIPTION_KEY = "a10bc58ac484478d9b3958b1742c3a03"
    USER_AGENT = "Meijer/100900000 okhttp/4.12.0 Dalvik/2.1.0 (Linux; U; Android 10; One Build/QQ3A.200705.002)"

    def __init__(self, bearer_token: str, cookies: dict = None):
        self.session = requests.Session()
        self.session.headers.update({
            "accept": "application/meijer.shoppingList.ShoppingList-v1.0+json",
            "authorization": f"Bearer {bearer_token}",
            "ocp-apim-subscription-key": self.SUBSCRIPTION_KEY,
            "user-agent": self.USER_AGENT,
            "accept-encoding": "gzip"
        })
        if cookies:
            self.session.cookies.update(cookies)

    def __repr__(self):
        #try:
        shopping_count = len(self.shopping_list)
        favorites_count = len(self.favorites)
        coupons_count = len(self.clipped_coupons)
        #except Exception:
        #    shopping_count = favorites_count = "?"
        return f"Meijer<list={shopping_count}, favorites={favorites_count}, coupons={coupons_count}>"

    def __del__(self):
        headers = {
            "content-type": "application/x-www-form-urlencoded",
            "user-agent": "Meijer/100900000 okhttp/4.12.0 Dalvik/2.1.0",
        }
        payload = {
            "client_id": self.client_id,
            "token": self.token,
        }
        try:
            requests.post(f"{self.AUTH_URL}/revoke", headers=headers, data=payload)
        except Exception:
            pass
        
    def get_shopping_list(self):
        url = f"{self.BASE_URL}/loyalty/shoppinglist/GetList"
        response = self.session.get(url)
        return response.json() if response.ok else response.text

    @property
    def shopping_list(self):
        return self.get_shopping_list().get("listItems", [])

    def delete_items(self):
        url = f"{self.BASE_URL}/loyalty/shoppinglist/DeleteAllListItems"
        self.session.headers.update({
            "accept": "application/vnd.meijer.listManagement.list-v1.0+json",
            "content-type": "application/vnd.meijer.listManagement.list-v1.0+json"
        })
        response = self.session.delete(url)
        return response.json() if response.ok else response.text

    def add_item(self, item_description: str = None, list_item_ids=None, favorite: bool = False, complete: bool = False):
        if item_description and list_item_ids:
            raise ValueError("Provide either item_description or list_item_ids, not both.")
        if not item_description and not list_item_ids:
            raise ValueError("Must provide item_description or list_item_ids.")
    
        if favorite and item_description:
            url = f"{self.BASE_URL}/loyalty/shoppinglist/AddFavoritesListItem"
            self.session.headers.update({
                "accept": "application/vnd.meijer.listManagement.favorites-v1.0+json",
                "content-type": "application/vnd.meijer.listManagement.favorites-v1.0+json"
            })
            payload = {
                "favoriteListItems": [{
                    "listItemId": 0,
                    "listItemTypeId": 0,
                    "itemDisplayOrder": 0,
                    "itemDescription": item_description,
                    "isItemInActiveList": False
                }]
            }
    
        else:
            url = f"{self.BASE_URL}/loyalty/shoppinglist/AddListItem"
            self.session.headers.update({
                "accept": "application/vnd.meijer.listManagement.list-v1.0+json",
                "content-type": "application/vnd.meijer.listManagement.list-v1.0+json"
            })
    
            if item_description:
                payload = {
                    "listItems": [{
                        "listItemId": 0,
                        "itemDescription": item_description,
                        "quantity": 1,
                        "listItemTypeId": 0,
                        "itemDisplayOrder": 0,
                        "storeId": 0,
                        "isComplete": complete,
                        "isFavorite": favorite,
                        "couponId": 0
                    }]
                }
            else:
                if isinstance(list_item_ids, int):
                    list_item_ids = [list_item_ids]
                payload = {
                    "listItems": [{"listItemId": i, "itemPartNumber": ""} for i in list_item_ids]
                }
    
        response = self.session.post(url, json=payload)
        return response.json() if response.ok else response.text


    def mark_complete(self, listItemId: int):
        url = f"{self.BASE_URL}/loyalty/shoppinglist/MarkAsCompleted/{listItemId}"
        self.session.headers.update({
            "accept": "application/vnd.meijer.listManagement.listItem-v1.0+json"
        })
        response = self.session.put(url)
        return response

    def mark_incomplete(self, listItemId: int):
        url = f"{self.BASE_URL}/loyalty/shoppinglist/MarkAsNotCompleted/{listItemId}"
        self.session.headers.update({
            "accept": "application/vnd.meijer.listManagement.listItem-v1.0+json"
        })
        response = self.session.put(url)
        return response
        
    def delete_items(self, completed: bool = True):
        endpoint = "/loyalty/shoppinglist/DeleteAllListItems"
        if completed:
            endpoint += "?deletecompleted=true"
        url = f"{self.BASE_URL}{endpoint}"
        self.session.headers.update({
            "accept": "application/vnd.meijer.listManagement.list-v1.0+json",
            "content-type": "application/vnd.meijer.listManagement.list-v1.0+json"
        })
        response = self.session.delete(url)
        return response

    def get_completion_count(self, complete: bool = True) -> int:
        shopping_list = self.get_shopping_list()
        return sum(1 for item in shopping_list.get("listItems", []) if item.get("isComplete") == complete)

    def refresh_tokens(self, refresh_token: str):
        url = f"{self.AUTH_URL}/oauth2/default/v1/token"
        headers = {
            "accept": "application/json",
            "user-agent": self.USER_AGENT,
            "content-type": "application/x-www-form-urlencoded",
        }
        data = {
            "client_id": "0oa1o8g9njWsUvwsx697",
            "grant_type": "refresh_token",
            "refresh_token": refresh_token,
        }
        response = requests.post(url, headers=headers, data=data, cookies=self.session.cookies)
        if response.ok:
            tokens = response.json()
            self.session.headers["authorization"] = f"Bearer {tokens['access_token']}"
            self.refresh_token = tokens.get("refresh_token")
        return response

    def get_favorites_list(self):
        url = f"{self.BASE_URL}/loyalty/shoppinglist/GetFavoritesList"
        headers = {
            "accept": "application/vnd.meijer.favorites-v1.0+json",
        }
        response = self.session.get(url, headers=headers)
        return response.json() if response.ok else response.text

    @property
    def favorites(self):
        return self.get_favorites_list().get("favoriteListItems", [])

    def delete_bulk_items(self, listItemIds):
        url = f"{self.BASE_URL}/loyalty/shoppinglist/DeleteBulkListItems"
        headers = {
            "accept": "application/vnd.meijer.listManagement.favorites-v1.0+json",
            "content-type": "application/vnd.meijer.listManagement.favorites-v1.0+json",
        }
        if isinstance(listItemIds, int):
            listItemIds = [listItemIds]
        data = {"listItemIds": listItemIds}
        response = self.session.post(url, headers=headers, json=data)
        return response

    def get_coupons(self, store_id=314, zip_code="46825"):
        url = f"{self.BASE_URL}/loyalty/mPerks/api/offers"
        self.session.headers.update({
            "accept": "application/vnd.meijer.digitalmperks.offers-v1.0+json",
            "content-type": "application/vnd.meijer.digitalmperks.offers-v1.0+json"
        })
        payload = {
            "sortType": "BySuggested",
            "pageSize": 9999,
            "currentPage": 1,
            "offerClass": 1,
            "searchCriteria": "",
            "zip": zip_code,
            "storeId": store_id,
            "ceilingCount": 0,
            "ceilingDuration": 0,
            "rewardCouponId": 0,
            "tagId": "",
            "getOfferCountPerDepartment": True,
            "upcList": [],
            "showClippedCoupons": False,
            "showOnlySpecialOffers": False,
            "showRedeemedOffers": False,
            "offerIds": [],
            "displayReasonFilters": []
        }
        response = self.session.post(url, json=payload)
        return response.json() if response.ok else response.text
    
    def clip_coupon(self, offer_id: int, store_id: int = 314, cart_is_active: bool = True):
        url = f"{self.BASE_URL}/loyalty/mPerks/api/offers/Clip"
        self.session.headers.update({
            "accept": "application/vnd.meijer.digitalmperks.clip-v1.0+json",
            "content-type": "application/vnd.meijer.digitalmperks.clip-v1.0+json"
        })
        payload = {
            "meijerOfferId": offer_id,
            "storeId": store_id,
            "cartIsActive": cart_is_active
        }
        response = self.session.post(url, json=payload)
        return response.json() if response.ok else response.text

    def unclip_coupon(self, offer_id: int, store_id: int = 314, cart_active: bool = True):
        url = f"{self.BASE_URL}/loyalty/mPerks/api/offers/Unclip"
        self.session.headers.update({
            "accept": "application/vnd.meijer.digitalmperks.unclip-v1.0+json",
            "content-type": "application/vnd.meijer.digitalmperks.unclip-v1.0+json"
        })
        payload = {
            "meijerOfferId": offer_id,
            "storeId": store_id,
            "cartIsActive": cart_active
        }
        response = self.session.post(url, json=payload)
        return response.json() if response.ok else response.text

    def get_clipped_coupons(self):
        url = f"{self.BASE_URL}/loyalty/mPerks/api/offers/ClippedOffers"
        self.session.headers.update({
            "accept": "application/vnd.meijer.digitalmperks.offers-v1.0+json",
            "content-type": "application/vnd.meijer.digitalmperks.offers-v1.0+json"
        })
        payload = {
            "sortType": "ByDepartmentSuggested",
            "pageSize": 9999,
            "currentPage": 1,
            "categoryId": "",
            "offerClass": 1,
            "upcList": [],
            "showOnlySpecialOffers": False,
            "showRedeemedOffers": False,
            "offerIds": [],
            "displayReasonFilters": []
        }
        response = self.session.post(url, json=payload)
        return response.json() if response.ok else response.text

    
    @property
    def coupons(self):
        return self.get_coupons()["listOfCoupons"]
        
    @property
    def clipped_coupons(self):
        return self.get_clipped_coupons()["listOfCoupons"]

In [74]:
token = "eyJraWQiOiJXMmxQc0g5Sy1lTWRoSExaeUZwSm9KTHFEamdMaFk4bEJjN21jd1A1UnZZIiwiYWxnIjoiUlMyNTYifQ.eyJ2ZXIiOjEsImp0aSI6IkFULm1YNXJDaHhNQzFwSXZ1TU1tRHhqUGJjZ3hoaEF5TUE2NHBPakVkY2RFa0Eub2FyMml4Z3phNThlN3dxSWE2OTciLCJpc3MiOiJodHRwczovL2lkLm1laWplci5jb20vb2F1dGgyL2RlZmF1bHQiLCJhdWQiOiJhcGk6Ly9kZWZhdWx0IiwiaWF0IjoxNzQyNzQxNDMyLCJleHAiOjE3NDI3NzAyMzIsImNpZCI6IjBvYTFvOGc5bmpXc1V2d3N4Njk3IiwidWlkIjoiMDB1cHVoMjU3N0prUDFCckY2OTciLCJzY3AiOlsib3BlbmlkIiwicHJvZmlsZSIsIm9mZmxpbmVfYWNjZXNzIl0sImF1dGhfdGltZSI6MTc0MjUwNDMxNywiaGFzX2RpZ2l0YWwiOiIxIiwiZWd1ZXN0X2lkIjoiMCIsInN1YiI6IjI2MjI4MjI4IiwiZGlnaXRhbF9pZCI6IjI2MjI4MjI4Iiwic2NvcGUiOlsib3BlbmlkIiwicHJvZmlsZSIsIm9mZmxpbmVfYWNjZXNzIl0sInJvbGVzIjpbImRpZ2l0YWw6YWNjb3VudHM6YWNjb3VudC5vd25lciJdLCJoYXNfbXBlcmtzIjoiMSIsIm1wZXJrc19zaG9wcGVyX2lkIjoiNDY4MTUxNTk1NTQiLCJtcGVya3NfZXh0X3Nob3BwZXJfaWQiOiI3NDUxMGMwZC1mNTAxLTQ5YmQtYTdlNC0yMDExY2E4MDRjOTUiLCJjbGllbnRfaWQiOiJva3RhIn0.FmqvzSbwtBysaa_2EjxPDoU7Iz5YKYToRSB2hnRGTjqhKoP2XRbryt5vnDhEojBqwH5lGl7EPrxCAGseC7fLYAfFT4wB8Alvy4zXVoFEOcCKUc4fM_HCrmjGbe8X1yCmVca2f3fBW6M3GKnL6jTNgKhDUeD_xwL2xmpo74GTN0GvkWadR5Yv0dFC6wjonAwzdUFRJ7Pehz6p1yKyo-LAZJ87xCKmEyoFAR7kaxddqfSkNm8K0Ri__Qz2trdFRHpZmwzFbnFsn0MDxEqr5-xwVEH0IhHtD6BL_pv6X-mr2prZvqYVex3A4dz-PsRCaBOYTlc5pEJ8leYxRKmfiqJK9w"
cookies = {"ROUTE": ".api-d7fbffc4d-9bg89"}

meijer = Meijer(bearer_token=token, cookies=cookies)
meijer

Meijer<list=10, favorites=10, coupons=159>

In [81]:
for coupon in meijer.clipped_coupons:
    meijer.unclip_coupon(coupon["offer"]["meijerOfferId"])

In [82]:
len(meijer.clipped_coupons)

0

In [84]:
meijer.coupons[0]

{'offer': {'meijerOfferId': 1116678815,
  'title': '15% off                                                     ',
  'titleColor': None,
  'description': 'Pet Dept.',
  'departments': [{'categoryID': 'SHOP',
    'categoryName': 'Online Deals',
    'subCategoryID': 'SHOP',
    'subCategoryName': 'Online Deals',
    'offerCountSubCategory': 0,
    'offerCountDepartment': 0,
    'isCustomCategory': True}],
  'category': {'segmentID': 'SHOP', 'segmentName': 'Online Deals'},
  'subcategory': {'segmentID': None, 'segmentName': None},
  'tags': ['SHOP', 'MHDPPetWeek032325'],
  'hatText': 'Online Deals',
  'hatColor': 2,
  'borderColor': 2,
  'isMeijerBuck': False,
  'showLargeImage': False,
  'imageURL': 'https://static.meijer.com/DigitalCoupon/20250323MHDPPets15P.png',
  'largeImageURL': None,
  'termsAndConditions': 'Valid Meijer.com/Meijer app Standard Delivery or Pickup Orders Only. Fulfilled by Shipt. Subject to applicable service fees. No cash back. Limit one coupon per customer. Produc

In [41]:
coupons.keys()

dict_keys(['couponCount', 'availableCouponCount', 'departmentCollection', 'listOfCoupons', 'responseCode', 'responseMessage', 'hasSpecialOffers', 'handpickedOffersCount'])

In [46]:
coupon["offer"]

{'meijerOfferId': 1116678835,
 'title': '$5.00 off                                                   ',
 'titleColor': None,
 'description': 'when you Buy $30 or more of Once Upon a Farm Organic Food reg 1.95 - 11.99. Baby Dept Only',
 'departments': [{'categoryID': '20250323inad',
   'categoryName': 'In Ad',
   'subCategoryID': '20250323inad',
   'subCategoryName': 'In Ad',
   'offerCountSubCategory': 0,
   'offerCountDepartment': 0,
   'isCustomCategory': True},
  {'categoryID': 'L5-000039',
   'categoryName': 'Baby',
   'subCategoryID': 'L4-000927',
   'subCategoryName': 'BABY FOOD AND GEAR',
   'offerCountSubCategory': 0,
   'offerCountDepartment': 0,
   'isCustomCategory': False}],
 'category': {'segmentID': '20250323inad', 'segmentName': 'In Ad'},
 'subcategory': {'segmentID': None, 'segmentName': None},
 'tags': ['20250323inad'],
 'hatText': 'In Ad',
 'hatColor': 2,
 'borderColor': 2,
 'isMeijerBuck': False,
 'showLargeImage': False,
 'imageURL': 'https://static.meijer.com/Media

In [47]:
for coupon in coupons["listOfCoupons"]:
    if coupon["offer"]['manufacturerCoupon']:
        break

In [49]:
coupon.keys()

dict_keys(['offer', 'isSuggested', 'isClipped', 'isAutoClipped', 'isHidden', 'isTargeted', 'couponInclusionGroupTag', 'couponExpirationGroupTag', 'isClippable', 'isSpecialOffer', 'redemptionDate'])

In [50]:
coupon["offer"]

{'meijerOfferId': 1216767105,
 'title': '$4.00 off 1                                                 ',
 'titleColor': None,
 'description': '"I and love and you" Cat Kibble, Any Size, Any Flavor',
 'departments': [{'categoryID': 'L5-000024',
   'categoryName': 'Pets',
   'subCategoryID': 'L4-000911',
   'subCategoryName': 'CAT FOOD AND TREATS',
   'offerCountSubCategory': 0,
   'offerCountDepartment': 0,
   'isCustomCategory': False}],
 'category': {'segmentID': 'L5-000024', 'segmentName': 'Pets'},
 'subcategory': {'segmentID': None, 'segmentName': None},
 'tags': [''],
 'hatText': None,
 'hatColor': 0,
 'borderColor': 0,
 'isMeijerBuck': False,
 'showLargeImage': False,
 'imageURL': 'https://static.meijer.com/Media/008/18336/0081833601333_1_A1C1_0600.png',
 'largeImageURL': None,
 'termsAndConditions': '*No cash back. Coupons are non-transferrable. Limit one use per coupon.',
 'manufacturerCoupon': True,
 'redemptionStartDate': '2025-03-02T00:00:00',
 'redemptionEndDate': '2025-03-29

In [53]:
meijer.clip_coupon(offer_id=coupon["offer"]["meijerOfferId"])

{'code': 0,
 'result': 'Success',
 'clipTimeStamp': '2025-03-23T21:46:29.9324679Z',
 'value': 0.0,
 'rewardType': 0}

In [55]:
for coupon in coupons["listOfCoupons"]:
    if coupon["offer"]['manufacturerCoupon']:
        meijer.clip_coupon(offer_id=coupon["offer"]["meijerOfferId"])

In [56]:
meijer.get_clipped_coupons()

{'couponCount': 160,
 'availableCouponCount': 160,
 'departmentCollection': [{'categoryID': '2024ExpressMondays',
   'categoryName': 'Express Mondays!',
   'subCategoryID': '2024ExpressMondays',
   'subCategoryName': 'Express Mondays!',
   'offerCountSubCategory': 1,
   'offerCountDepartment': 1,
   'isCustomCategory': True},
  {'categoryID': '20250323inad',
   'categoryName': 'In Ad',
   'subCategoryID': '20250323inad',
   'subCategoryName': 'In Ad',
   'offerCountSubCategory': 19,
   'offerCountDepartment': 19,
   'isCustomCategory': True},
  {'categoryID': 'GasStationOnly',
   'categoryName': 'Meijer Express Only',
   'subCategoryID': 'GasStationOnly',
   'subCategoryName': 'Meijer Express Only',
   'offerCountSubCategory': 1,
   'offerCountDepartment': 1,
   'isCustomCategory': True},
  {'categoryID': 'L5-000070',
   'categoryName': 'Adult Beverages',
   'subCategoryID': 'L4-000679',
   'subCategoryName': 'Liquor',
   'offerCountSubCategory': 2,
   'offerCountDepartment': 2,
   'is

In [57]:
coupons = meijer.get_clipped_coupons()

In [59]:
coupons["listOfCoupons"]

[{'offer': {'meijerOfferId': 1216774391,
   'title': 'Free                                                        ',
   'titleColor': None,
   'description': 'Bang Any Means Orange 16 oz (Meijer Express Only)',
   'departments': [{'categoryID': '2024ExpressMondays',
     'categoryName': 'Express Mondays!',
     'subCategoryID': '2024ExpressMondays',
     'subCategoryName': 'Express Mondays!',
     'offerCountSubCategory': 0,
     'offerCountDepartment': 0,
     'isCustomCategory': True},
    {'categoryID': 'L5-000108',
     'categoryName': 'Gas Station',
     'subCategoryID': 'L4-000848',
     'subCategoryName': 'Convenience Store',
     'offerCountSubCategory': 0,
     'offerCountDepartment': 0,
     'isCustomCategory': False}],
   'category': {'segmentID': '2024ExpressMondays',
    'segmentName': 'Express Mondays!'},
   'subcategory': {'segmentID': None, 'segmentName': None},
   'tags': ['2024ExpressMondays'],
   'hatText': 'Express Mondays!',
   'hatColor': 1,
   'borderColor': 1,
 

In [60]:
len(coupons["listOfCoupons"])

160

In [61]:
ccoupons = coupons["listOfCoupons"]

In [63]:
meijer.unclip_coupon(ccoupons[0]["offer"]["meijerOfferId"])

{'code': 0,
 'result': 'Success',
 'clipTimeStamp': None,
 'value': 0.0,
 'rewardType': 0}

In [64]:
coupons = meijer.get_clipped_coupons()
coupons

{'couponCount': 159,
 'availableCouponCount': 159,
 'departmentCollection': [{'categoryID': '20250323inad',
   'categoryName': 'In Ad',
   'subCategoryID': '20250323inad',
   'subCategoryName': 'In Ad',
   'offerCountSubCategory': 19,
   'offerCountDepartment': 19,
   'isCustomCategory': True},
  {'categoryID': 'GasStationOnly',
   'categoryName': 'Meijer Express Only',
   'subCategoryID': 'GasStationOnly',
   'subCategoryName': 'Meijer Express Only',
   'offerCountSubCategory': 1,
   'offerCountDepartment': 1,
   'isCustomCategory': True},
  {'categoryID': 'L5-000070',
   'categoryName': 'Adult Beverages',
   'subCategoryID': 'L4-000679',
   'subCategoryName': 'Liquor',
   'offerCountSubCategory': 2,
   'offerCountDepartment': 2,
   'isCustomCategory': False},
  {'categoryID': 'L5-000039',
   'categoryName': 'Baby',
   'subCategoryID': 'L4-000519',
   'subCategoryName': 'Baby Needs',
   'offerCountSubCategory': 4,
   'offerCountDepartment': 4,
   'isCustomCategory': False},
  {'catego

[{'offer': {'meijerOfferId': 1216774482,
   'title': '$3.00 off 2                                                 ',
   'titleColor': None,
   'description': "Adult Crest Paste 2.4 oz. or More, Crest Kids Advanced OR Burt's Bees Adult Paste 4.0 oz. or More, Crest, Scope OR Oral-B Mouthwash 473 mL. or Larger, Scope Squeez, Oral-B Adult Manual Brush, Expandable/Oral-B Glide Floss OR Interdental Picks/Brush, Fixodent Adhesive 1.4 oz. or Larger (excludes Crest Cavity, Baking Soda, Tartar, More Free Packs, other Kids Variants. Oral-B Essential Brushes, Daily Clean, Complete 1 ct. Brushes, Essential, Satin Floss & Oral-B Fresh Mint Picks and trial/travel size)",
   'departments': [{'categoryID': '20250323inad',
     'categoryName': 'In Ad',
     'subCategoryID': '20250323inad',
     'subCategoryName': 'In Ad',
     'offerCountSubCategory': 0,
     'offerCountDepartment': 0,
     'isCustomCategory': True},
    {'categoryID': 'L5-000002',
     'categoryName': 'Health Care',
     'subCategoryID

In [8]:
for _ in range(10):
    meijer.add_item(pronounceable_passwd(10), favorite=True)
    meijer.add_item(pronounceable_passwd(10), favorite=False)

In [10]:
len(meijer.shopping_list) 

10

In [11]:
len(meijer.favorites)

10

In [12]:
meijer.delete_items(completed=True)

<Response [200]>

In [13]:
len(meijer.shopping_list) 

10

In [14]:
meijer.delete_items(completed=False)

<Response [200]>

In [15]:
len(meijer.shopping_list)

0

In [16]:
favorites = meijer.favorites  
active_favorite_ids = [f["listItemId"] for f in favorites if not f["isItemInActiveList"]]
active_favorite_ids

[269782574,
 269782572,
 269782570,
 269782567,
 269782565,
 269782563,
 269782561,
 269782559,
 269782557,
 269782555]

In [17]:
meijer.add_item(list_item_ids=active_favorite_ids)

{'updateConfirmations': [{'listItemId': 269783065,
   'listItemTypeId': 1,
   'itemDisplayOrder': 1,
   'itemPartNumber': '',
   'itemDescription': 'tuardendwo',
   'quantity': 1,
   'storeId': None,
   'notes': '',
   'couponId': None,
   'isComplete': False,
   'isFavorite': True,
   'listingId': None,
   'promotionEnd': None,
   'promotionStart': None},
  {'listItemId': 269783066,
   'listItemTypeId': 1,
   'itemDisplayOrder': 2,
   'itemPartNumber': '',
   'itemDescription': 'aryansteew',
   'quantity': 1,
   'storeId': None,
   'notes': '',
   'couponId': None,
   'isComplete': False,
   'isFavorite': True,
   'listingId': None,
   'promotionEnd': None,
   'promotionStart': None},
  {'listItemId': 269783067,
   'listItemTypeId': 1,
   'itemDisplayOrder': 3,
   'itemPartNumber': '',
   'itemDescription': 'anderypocy',
   'quantity': 1,
   'storeId': None,
   'notes': '',
   'couponId': None,
   'isComplete': False,
   'isFavorite': True,
   'listingId': None,
   'promotionEnd': Non

In [18]:
len(meijer.shopping_list)

10

In [20]:
for item in meijer.shopping_list:
    meijer.mark_complete(item['listItemId'])

In [25]:
coupons = meijer.get_coupons()

In [27]:
coupons.keys()

dict_keys(['couponCount', 'availableCouponCount', 'departmentCollection', 'listOfCoupons', 'responseCode', 'responseMessage', 'hasSpecialOffers', 'handpickedOffersCount'])

In [29]:
coupons['listOfCoupons'][0]

{'offer': {'meijerOfferId': 1116678815,
  'title': '15% off                                                     ',
  'titleColor': None,
  'description': 'Pet Dept.',
  'departments': [{'categoryID': 'SHOP',
    'categoryName': 'Online Deals',
    'subCategoryID': 'SHOP',
    'subCategoryName': 'Online Deals',
    'offerCountSubCategory': 0,
    'offerCountDepartment': 0,
    'isCustomCategory': True}],
  'category': {'segmentID': 'SHOP', 'segmentName': 'Online Deals'},
  'subcategory': {'segmentID': None, 'segmentName': None},
  'tags': ['SHOP', 'MHDPPetWeek032325'],
  'hatText': 'Online Deals',
  'hatColor': 2,
  'borderColor': 2,
  'isMeijerBuck': False,
  'showLargeImage': False,
  'imageURL': 'https://static.meijer.com/DigitalCoupon/20250323MHDPPets15P.png',
  'largeImageURL': None,
  'termsAndConditions': 'Valid Meijer.com/Meijer app Standard Delivery or Pickup Orders Only. Fulfilled by Shipt. Subject to applicable service fees. No cash back. Limit one coupon per customer. Produc

In [ ]:
coupons["listOfCoupons"]

In [198]:
meijer.add_item(pronounceable_passwd(10), favorite=True)

{'updateConfirmations': [{'listItemId': 269781301,
   'listItemTypeId': 1,
   'itemDisplayOrder': 3,
   'itemPartNumber': '',
   'itemDescription': 'alzarrypto',
   'isItemInActiveList': False}]}

In [199]:
meijer

Meijer<list=7, favorites=3>

In [201]:
meijer.add_item(pronounceable_passwd(10), favorite=False)

{'updateConfirmations': [{'listItemId': 269781332,
   'listItemTypeId': 1,
   'itemDisplayOrder': 9,
   'itemPartNumber': '',
   'itemDescription': 'debruntegh',
   'quantity': 1,
   'storeId': None,
   'notes': '',
   'couponId': None,
   'isComplete': False,
   'isFavorite': False,
   'listingId': None,
   'promotionEnd': None,
   'promotionStart': None}]}

In [203]:
meijer

Meijer<list=8, favorites=3>

In [204]:
meijer.favorites

[{'listItemId': 269781301,
  'listItemTypeId': 1,
  'itemDisplayOrder': 3,
  'itemPartNumber': '',
  'itemDescription': 'alzarrypto',
  'isItemInActiveList': False},
 {'listItemId': 269780853,
  'listItemTypeId': 1,
  'itemDisplayOrder': 2,
  'itemPartNumber': '',
  'itemDescription': 'quadulsion',
  'isItemInActiveList': True},
 {'listItemId': 269780697,
  'listItemTypeId': 1,
  'itemDisplayOrder': 1,
  'itemPartNumber': '',
  'itemDescription': 'foo',
  'isItemInActiveList': True}]

In [205]:
favorites = meijer.favorites  
active_favorite_ids = [f["listItemId"] for f in favorites if not f["isItemInActiveList"]]
active_favorite_ids

[269781301]

In [206]:
meijer.add_item(list_item_ids=active_favorite_ids)

{'updateConfirmations': [{'listItemId': 269781645,
   'listItemTypeId': 1,
   'itemDisplayOrder': 10,
   'itemPartNumber': '',
   'itemDescription': 'alzarrypto',
   'quantity': 1,
   'storeId': None,
   'notes': '',
   'couponId': None,
   'isComplete': False,
   'isFavorite': True,
   'listingId': None,
   'promotionEnd': None,
   'promotionStart': None}]}

In [193]:
meijer.add_item(pronounceable_passwd(10), favorite=False)

{'updateConfirmations': [{'listItemId': 269780863,
   'listItemTypeId': 1,
   'itemDisplayOrder': 7,
   'itemPartNumber': '',
   'itemDescription': 'fluminiunt',
   'quantity': 1,
   'storeId': None,
   'notes': '',
   'couponId': None,
   'isComplete': False,
   'isFavorite': False,
   'listingId': None,
   'promotionEnd': None,
   'promotionStart': None}]}

In [160]:
list_item_ids = [item['listItemId'] for item in meijer.favorites]
list_item_ids

[]

In [157]:
response = meijer.delete_bulk_items(list_item_ids)

In [158]:
response

<Response [200]>

In [124]:
meijer.add_item(pronounceable_passwd(10), favorite=True)

{'updateConfirmations': [{'listItemId': 269771773,
   'listItemTypeId': 1,
   'itemDisplayOrder': 4,
   'itemPartNumber': '',
   'itemDescription': 'faccasprie',
   'isItemInActiveList': False}]}

In [125]:
meijer.get_favorites_list()

{'listId': 24189647,
 'listName': 'favorite list items for mPerks Web and Mobile',
 'listTypeId': 2,
 'totalCount': 4,
 'favoriteListItems': [{'listItemId': 269771773,
   'listItemTypeId': 1,
   'itemDisplayOrder': 4,
   'itemPartNumber': '',
   'itemDescription': 'faccasprie',
   'isItemInActiveList': False},
  {'listItemId': 269771771,
   'listItemTypeId': 1,
   'itemDisplayOrder': 3,
   'itemPartNumber': '',
   'itemDescription': 'previngina',
   'isItemInActiveList': False},
  {'listItemId': 269771766,
   'listItemTypeId': 1,
   'itemDisplayOrder': 2,
   'itemPartNumber': '',
   'itemDescription': 'pissustrou',
   'isItemInActiveList': False},
  {'listItemId': 269771763,
   'listItemTypeId': 1,
   'itemDisplayOrder': 1,
   'itemPartNumber': '',
   'itemDescription': 'carilannic',
   'isItemInActiveList': False}]}

In [103]:
list_item_ids = [item['listItemId'] for item in meijer.get_favorites_list()['favoriteListItems']]
list_item_ids

[269771016]

In [104]:
response = meijer.delete_bulk_items(list_item_ids)

In [207]:
meijer.delete_items(completed=False)

<Response [200]>

In [ ]:
meijer.delete_bulk_items

In [63]:
meijer.refresh_tokens("n5rzx4sjmR942EEPaLLmkoSArZmBONZTx9AQPbqIOjw")

<Response [400]>

In [48]:
for _ in range(10):
    meijer.add_item(fake.name_male(), favorite=False)

In [49]:
len(meijer.get_shopping_list()["listItems"])

10

In [50]:
for idx, item in enumerate(shopping_list["listItems"]):
    meijer.mark_complete(item["listItemId"])

In [52]:
response = meijer.mark_incomplete(item["listItemId"])

In [53]:
response.status_code

205

In [41]:
meijer.get_shopping_list()

{'listId': 24176630,
 'listName': 'Meijer Shopping List for mPerks Web and Mobile',
 'listTypeId': 1,
 'totalCount': 10,
 'listItems': [{'listItemId': 269720790,
   'listItemTypeId': 1,
   'itemDisplayOrder': 10,
   'itemPartNumber': '',
   'itemDescription': 'James Carpenter',
   'quantity': 1,
   'storeId': None,
   'notes': '',
   'couponId': None,
   'isComplete': False,
   'isFavorite': False,
   'listingId': None,
   'promotionEnd': None,
   'promotionStart': None},
  {'listItemId': 269720788,
   'listItemTypeId': 1,
   'itemDisplayOrder': 9,
   'itemPartNumber': '',
   'itemDescription': 'David Schultz',
   'quantity': 1,
   'storeId': None,
   'notes': '',
   'couponId': None,
   'isComplete': False,
   'isFavorite': False,
   'listingId': None,
   'promotionEnd': None,
   'promotionStart': None},
  {'listItemId': 269720787,
   'listItemTypeId': 1,
   'itemDisplayOrder': 8,
   'itemPartNumber': '',
   'itemDescription': 'Bryce Garner',
   'quantity': 1,
   'storeId': None,
   '

In [39]:
meijer.get_completion_count()

0

In [145]:
for item in shopping_list["listItems"]:
    meijer.mark_incomplete(item["listItemId"])

In [146]:
assert meijer.get_completion_count()==0

In [156]:
response = 

In [157]:
response.status_code

200

In [162]:
meijer.delete_items(completed=False)

True

In [178]:
meijer = Meijer(bearer_token=token, cookies=cookies)
meijer.delete_items(completed=False)

True

In [179]:
meijer.get_shopping_list()['listItems']

[]

In [180]:
for _ in range(10):
    meijer.add_item(fake.name_male(), favorite=False)

In [181]:
assert len(meijer.get_shopping_list()['listItems'])==10

In [182]:
meijer.mark_complete(item["listItemId"])

False

In [183]:
for idx, item in enumerate(shopping_list["listItems"]):
    break
    

In [184]:
meijer.mark_complete(item["listItemId"])

False

In [ ]:
    meijer.mark_complete(item["listItemId"])

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [58]:
"eyJraWQiOiJXMmxQc0g5Sy1lTWRoSExaeUZwSm9KTHFEamdMaFk4bEJjN21jd1A1UnZZIiwiYWxnIjoiUlMyNTYifQ.eyJ2ZXIiOjEsImp0aSI6IkFULm1YNXJDaHhNQzFwSXZ1TU1tRHhqUGJjZ3hoaEF5TUE2NHBPakVkY2RFa0Eub2FyMml4Z3phNThlN3dxSWE2OTciLCJpc3MiOiJodHRwczovL2lkLm1laWplci5jb20vb2F1dGgyL2RlZmF1bHQiLCJhdWQiOiJhcGk6Ly9kZWZhdWx0IiwiaWF0IjoxNzQyNzQxNDMyLCJleHAiOjE3NDI3NzAyMzIsImNpZCI6IjBvYTFvOGc5bmpXc1V2d3N4Njk3IiwidWlkIjoiMDB1cHVoMjU3N0prUDFCckY2OTciLCJzY3AiOlsib3BlbmlkIiwicHJvZmlsZSIsIm9mZmxpbmVfYWNjZXNzIl0sImF1dGhfdGltZSI6MTc0MjUwNDMxNywiaGFzX2RpZ2l0YWwiOiIxIiwiZWd1ZXN0X2lkIjoiMCIsInN1YiI6IjI2MjI4MjI4IiwiZGlnaXRhbF9pZCI6IjI2MjI4MjI4Iiwic2NvcGUiOlsib3BlbmlkIiwicHJvZmlsZSIsIm9mZmxpbmVfYWNjZXNzIl0sInJvbGVzIjpbImRpZ2l0YWw6YWNjb3VudHM6YWNjb3VudC5vd25lciJdLCJoYXNfbXBlcmtzIjoiMSIsIm1wZXJrc19zaG9wcGVyX2lkIjoiNDY4MTUxNTk1NTQiLCJtcGVya3NfZXh0X3Nob3BwZXJfaWQiOiI3NDUxMGMwZC1mNTAxLTQ5YmQtYTdlNC0yMDExY2E4MDRjOTUiLCJjbGllbnRfaWQiOiJva3RhIn0.FmqvzSbwtBysaa_2EjxPDoU7Iz5YKYToRSB2hnRGTjqhKoP2XRbryt5vnDhEojBqwH5lGl7EPrxCAGseC7fLYAfFT4wB8Alvy4zXVoFEOcCKUc4fM_HCrmjGbe8X1yCmVca2f3fBW6M3GKnL6jTNgKhDUeD_xwL2xmpo74GTN0GvkWadR5Yv0dFC6wjonAwzdUFRJ7Pehz6p1yKyo-LAZJ87xCKmEyoFAR7kaxddqfSkNm8K0Ri__Qz2trdFRHpZmwzFbnFsn0MDxEqr5-xwVEH0IhHtD6BL_pv6X-mr2prZvqYVex3A4dz-PsRCaBOYTlc5pEJ8leYxRKmfiqJK9w"

'eyJraWQiOiJXMmxQc0g5Sy1lTWRoSExaeUZwSm9KTHFEamdMaFk4bEJjN21jd1A1UnZZIiwiYWxnIjoiUlMyNTYifQ.eyJ2ZXIiOjEsImp0aSI6IkFULm1YNXJDaHhNQzFwSXZ1TU1tRHhqUGJjZ3hoaEF5TUE2NHBPakVkY2RFa0Eub2FyMml4Z3phNThlN3dxSWE2OTciLCJpc3MiOiJodHRwczovL2lkLm1laWplci5jb20vb2F1dGgyL2RlZmF1bHQiLCJhdWQiOiJhcGk6Ly9kZWZhdWx0IiwiaWF0IjoxNzQyNzQxNDMyLCJleHAiOjE3NDI3NzAyMzIsImNpZCI6IjBvYTFvOGc5bmpXc1V2d3N4Njk3IiwidWlkIjoiMDB1cHVoMjU3N0prUDFCckY2OTciLCJzY3AiOlsib3BlbmlkIiwicHJvZmlsZSIsIm9mZmxpbmVfYWNjZXNzIl0sImF1dGhfdGltZSI6MTc0MjUwNDMxNywiaGFzX2RpZ2l0YWwiOiIxIiwiZWd1ZXN0X2lkIjoiMCIsInN1YiI6IjI2MjI4MjI4IiwiZGlnaXRhbF9pZCI6IjI2MjI4MjI4Iiwic2NvcGUiOlsib3BlbmlkIiwicHJvZmlsZSIsIm9mZmxpbmVfYWNjZXNzIl0sInJvbGVzIjpbImRpZ2l0YWw6YWNjb3VudHM6YWNjb3VudC5vd25lciJdLCJoYXNfbXBlcmtzIjoiMSIsIm1wZXJrc19zaG9wcGVyX2lkIjoiNDY4MTUxNTk1NTQiLCJtcGVya3NfZXh0X3Nob3BwZXJfaWQiOiI3NDUxMGMwZC1mNTAxLTQ5YmQtYTdlNC0yMDExY2E4MDRjOTUiLCJjbGllbnRfaWQiOiJva3RhIn0.FmqvzSbwtBysaa_2EjxPDoU7Iz5YKYToRSB2hnRGTjqhKoP2XRbryt5vnDhEojBqwH5lGl7EPrxCAGseC7fLYAfFT4wB8Alv